In [ ]:
# Récupérer la page : requests

import requests

URL = "https://books.toscrape.com/catalogue/page-1.html"
ENTETES = {"User-Agent": "bootcamp-data/1.0 (bourreauclemence@gmail.com)"}

r = requests.get(URL, headers=ENTETES, timeout=10)
r.encoding = "utf-8"

print("status   :", r.status_code)
print("longueur :", len(r.text), "caractères")
print("début    :", r.text[:60])

status   : 200
longueur : 50449 caractères
début    : 

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" 


In [3]:
print("le prix est present :", "£51.77" in r.text)
print("le titre est present:", "A Light in the Attic" in r.text)

le prix est present : True
le titre est present: True


In [4]:
# Appliquer tes sélecteurs : BeautifulSoup

from bs4 import BeautifulSoup

soup = BeautifulSoup(r.text, "html.parser")

fiches = soup.select("article.product_pod")
print("fiches trouvées :", len(fiches))

fiches trouvées : 20


In [7]:
fiche = fiches[0]

prix = fiche.select_one("p.price_color").get_text(strip=True)

lien_h3 = fiche.select_one("h3 a")
titre = lien_h3["title"]
href = lien_h3["href"]

print("prix     :", prix)
print("titre    :", titre)
print("href     :", href)

prix     : £51.77
titre    : A Light in the Attic
href     : a-light-in-the-attic_1000/index.html


In [8]:
classes = fiche.select_one("p.star-rating")["class"]
print("classes :", classes)

classes : ['star-rating', 'Three']


In [9]:
NOTES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

note = NOTES[classes[1]]
print("note :", note)

note : 3


In [10]:
from urllib.parse import urljoin

print(urljoin(URL, href))

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


In [11]:
def lire_fiche(fiche, url_page):
    lien = fiche.select_one("h3 a")
    classes = fiche.select_one("p.star-rating")["class"]
    return {
        "titre": lien["title"],
        "prix": fiche.select_one("p.price_color").get_text(strip=True),
        "dispo": fiche.select_one("p.instock").get_text(strip=True),
        "note": NOTES[classes[1]],
        "url": urljoin(url_page, lien["href"]),
    }

print(lire_fiche(fiches[0], URL))

{'titre': 'A Light in the Attic', 'prix': '£51.77', 'dispo': 'In stock', 'note': 3, 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}


In [12]:
# Plusieurs pages : suivre la pagination
suivant = soup.select_one("li.next a")
print(suivant["href"] if suivant else "pas de page suivante")

page-2.html


In [13]:
import time

def collecter_toutes_les_pages(url_depart, pages_max=3):
    url = url_depart
    livres = []
    page = 0

    while url and page < pages_max:
        page += 1
        r = requests.get(url, headers=ENTETES, timeout=10)
        r.encoding = "utf-8"
        if r.status_code != 200:
            print(f"    page {page} : status {r.status_code}, on s'arrête")
            break
    
        soup = BeautifulSoup(r.text, "html.parser")
        fiches = soup.select("article.product_pod")
        livres += [lire_fiche(f, url) for f in fiches]
        print(f"    page {page} : {len(fiches)} livres (total {len(livres)})")

        suivant = soup.select_one("li.next a")
        url = urljoin(url, suivant["href"]) if suivant else None

        time.sleep(1)
    return livres

livres = collecter_toutes_les_pages("https://books.toscrape.com/catalogue/page-1.html")
print("collecte :", len(livres), "livres")

    page 1 : 20 livres (total 20)
    page 2 : 20 livres (total 40)
    page 3 : 20 livres (total 60)
collecte : 60 livres


In [14]:
# Le second niveau : suivre les liens
def lire_detail(url):
    r = requests.get(url, headers=ENTETES, timeout=10)
    r.encoding = "utf-8"
    if r.status_code != 200:
        return {"categorie": None, "stock": None}
    
    soup = BeautifulSoup(r.text, "html.parser")

    infos = {}
    for ligne in soup.select("table.table-striped tr"):
        cle = ligne.select_one("th")
        val = ligne.select_one("td")
        if cle and val:
            infos[cle.get_text(strip=True)] = val.get_text(strip=True)
    
    miettes = soup.select("ul.breadcrumb li a")
    categorie = miettes[-1].get_text(strip=True) if miettes else None

    return {
        "categorie": categorie,
        "stock": infos.get("Availability"),
        "upc": infos.get("UPC"),
    }

url_test = "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html"
print(lire_detail(url_test))

{'categorie': 'Poetry', 'stock': 'In stock (22 available)', 'upc': 'a897fe39b1053632'}


In [15]:
import re

def nombre_en_stock(texte):
    if not texte:
        return None
    trouve = re.search(r"\((\d+) available\)", texte)
    return int(trouve.group(1)) if trouve else 0

print(nombre_en_stock("In stock (22 available)"))
print(nombre_en_stock("Out of stock"))
print(nombre_en_stock(None))

22
0
None
